# nb136 drafts · FQ3 + FQ2
Follow-ups to nb136 Q3 (where position lives in the input) and Q2 / Q1 (how the bands are used), on run **decres-rescale-aug10-v1** (`src9/arch/transformer4.py`, eager fp32, IoU @ 0.08).
Same fixed sets as Q1 (`subset720.npz`): 60 each of candle / red-cube / soap dispenser / vase per box, **240 per box**, train and ID eval.
Caches: `scripts/nb136_fq3.py SET` → `fq3_{SET}.npz`, `scripts/nb136_fq2.py SET` → `fq2_{SET}.npz`. Relative drop = Σ(clean − edited) / Σ clean per box (same `rel` as nb136 Q1), shown as **% of clean IoU**.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.signal import savgol_filter, find_peaks
from scipy.ndimage import maximum_filter1d
ROOT = Path.cwd().parents[1] if Path.cwd().name == "drafts_136" else Path.cwd()
CACHE = ROOT / "runs/_src9_cache/nb136"
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
BX, SETS = ["plastic", "gastro", "cardboard"], ["train", "eval"]
BOXC, CMAP = {"plastic": "tab:blue", "gastro": "tab:green", "cardboard": "tab:orange"}, {"plastic": "Blues", "gastro": "Greens", "cardboard": "Oranges"}
HZ, FHZ = 100 + (np.arange(18) * 65 + 32) * 900 / 1169, np.linspace(100, 1000, 1170)       # band centres (50 Hz bands), bin Hz
Q = {s: np.load(CACHE / f"q1_{s}.npz") for s in SETS}; F3 = {s: np.load(CACHE / f"fq3_{s}.npz") for s in SETS}; F2 = {s: np.load(CACHE / f"fq2_{s}.npz") for s in SETS}
for s in SETS: assert (F3[s]["rows"] == Q[s]["rows"]).all() and (F2[s]["rows"] == Q[s]["rows"]).all()
c0 = {s: {b: Q[s]["clean"][Q[s]["box"] == b].mean() for b in BX} for s in SETS}
absd = lambda r, cl: r * np.clip(cl, 1e-3, None)                                   # caches store per-sample (clean − e) / max(clean, 1e-3) -> back to clean − e
print({s: {b: round(float(c0[s][b]), 3) for b in BX} for s in SETS})

## FQ3 · What spectral change carries position?
For each of the 18 bands (65 bins, 50 Hz), one counterfactual edit of **that band only**, applied to |X| before the model's per-sample z-score (`transformer4.py:139`), so the normalisation still runs:
- **pattern** (across-laser pattern removed): every laser gets the band's laser-mean spectrum, per axis;
- **amp** (amplitude removed): each laser's band rescaled to the band's mean level over lasers, shape kept;
- **shift ±1, ±2**: the band rolled by k bins (0.77 Hz / bin), edge-padded;
- **smooth**: 9-bin boxcar (≈ 7 Hz) within the band, edge-padded;
- **key-mask**: the band token dropped through the model's key mask (nb136 Q1 cache `q1_{SET}.npz`).

Note pattern ⊃ amp: removing the across-laser pattern also removes the across-laser level differences. Shift and smooth keep the spatial pattern but move / blur the spectral lines.

In [ ]:
ED = ["pattern", "amp", "shift-2", "shift-1", "shift+1", "shift+2", "smooth", "keymask"]
def drops(s):   # {box: (8 edits, 18 bands)} relative IoU drop, % of clean
    r, box, cl = F3[s]["drop"], F3[s]["box"], F3[s]["clean"]; out = {}
    for b in BX:
        m = box == b; d = np.stack([absd(r[i][:, m], cl[m]).mean(1) for i in range(7)] + [Q[s]["mask_b"][m].mean(0)])
        out[b] = 100 * d / c0[s][b]
    return out
DR = {s: drops(s) for s in SETS}
fig, ax = plt.subplots(2, 3, figsize=(13, 6.6), constrained_layout=True)
for i, s in enumerate(SETS):
    for j, b in enumerate(BX):
        v = np.clip(DR[s][b], 0, None); h = ax[i, j].imshow(v, cmap=CMAP[b], vmin=0, aspect="auto")
        plt.colorbar(h, ax=ax[i, j], fraction=.04, pad=.01, label="% of clean IoU lost")
        ax[i, j].set_yticks(range(8), ED); ax[i, j].set_xticks(range(0, 18, 2), [f"{h:.0f}" for h in HZ[::2]], rotation=60, fontsize=8)
        k = np.unravel_index(DR[s][b].argmax(), DR[s][b].shape); ax[i, j].set_title(f"{s} · {b} (clean IoU {c0[s][b]:.2f}) · max {DR[s][b].max():.1f}% = {ED[k[0]]} @ {HZ[k[1]]:.0f} Hz", fontsize=9)
        ax[i, j].set_xlabel("band centre (Hz)" if i else "")
fig.suptitle("FQ3 · relative IoU drop per band × edit (one band edited at a time; negatives clipped at 0)")
plt.show()

In [ ]:
import pandas as pd
rows = []
for s in SETS:
    for b in BX:
        d = DR[s][b]; sh1, sh2 = d[[3, 4]].mean(0), d[[2, 5]].mean(0)
        r = {"set": s, "box": b}
        for nm, v in [("pattern", d[0]), ("amp", d[1]), ("shift±1", sh1), ("shift±2", sh2), ("smooth", d[6]), ("keymask", d[7])]:
            r[f"{nm} Σ"] = v.sum().round(1); r[f"{nm} max"] = f"{v.max():.1f}@{HZ[v.argmax()]:.0f}"
        rows.append(r)
T3 = pd.DataFrame(rows).set_index(["set", "box"]); T3

In [ ]:
# where in frequency does each edit act: importance-weighted mean band Hz, and rank agreement with key-masking (Spearman over the 18 bands)
rows = []
for s in SETS:
    for b in BX:
        d = np.clip(DR[s][b], 0, None); r = {"set": s, "box": b}
        for i, nm in enumerate(ED[:-1]): r[f"{nm} ρ(mask)"] = round(stats.spearmanr(DR[s][b][i], DR[s][b][7])[0], 2)
        for i, nm in [(0, "pattern"), (1, "amp"), (2, "shift-2"), (6, "smooth"), (7, "keymask")]: r[f"{nm} Hz̄"] = int((d[i] * HZ).sum() / d[i].sum())
        rows.append(r)
pd.DataFrame(rows).set_index(["set", "box"])

### FQ3 · gradient × input at full 1170-bin resolution
∂(soft Dice)/∂|X| · |X| per sample (soft Dice = 2Σpy / (Σp + Σy), p = sigmoid of the mask logits, the IoU proxy), **|g·x| summed over the 100 lasers and 2 axes**, averaged over the box's 240 samples (absolute, so position-dependent signs of opposite laser groups do not cancel). Ticks = the box's **relaxed resonances** from nb136 Q1b (same rule: savgol σ(f), prominence ≥ 2 × MAD from a 60 Hz baseline, present in ≥ 50% of captures of an (object, speaker) group), computed on the same set.

In [ ]:
def peaks(sig, k_prom=2., baseline_hz=60., distance_hz=5., smooth_hz=5.):    # nb136 Q1b / nb109 §12 rule, relaxed defaults
    nb = lambda hz: max(1, int(round(hz * 1.3))); odd = lambda n: n + 1 - n % 2
    sig = savgol_filter(sig, odd(2 * nb(smooth_hz) + 1), 3); r = sig - savgol_filter(sig, odd(2 * nb(baseline_hz) + 1), 3)
    return find_peaks(sig, distance=nb(distance_hz), prominence=max(1e-12, k_prom * 1.4826 * np.median(np.abs(r - np.median(r)))), wlen=odd(2 * nb(baseline_hz) + 1))[0]
RES = {}
for s in SETS:
    q = Q[s]
    for b in BX:
        O, Sg = [], []
        for o in np.unique(q["obj"]):
            for sp in np.unique(q["spk"]):
                g = (q["box"] == b) & (q["obj"] == o) & (q["spk"] == sp)
                if g.sum() < 3: continue
                hit = np.zeros(q["sigma"][g].shape, np.uint8)
                for i, x in enumerate(q["sigma"][g]): hit[i, peaks(x)] = 1
                O.append(maximum_filter1d(hit, 7, axis=1).mean(0)); Sg.append(q["sigma"][g].mean(0))
        occ = np.mean(O, 0); RES[s, b] = np.array([k for k in peaks(np.mean(Sg, 0)) if occ[k] >= .5])
print({k: len(v) for k, v in RES.items()})
GX = {(s, b): F3[s]["agx"][F3[s]["box"] == b].sum(-1).mean(0) for s in SETS for b in BX}    # (1170,) mean |g·x|
fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
for j, b in enumerate(BX):
    for s, ls in zip(SETS, ["-", "--"]): g = GX[s, b]; ax[j].plot(FHZ, g / g.sum() * 100, ls, color=BOXC[b], lw=1 if s == "train" else .8, alpha=1 if s == "train" else .7, label=s)
    for k in RES["train", b]: ax[j].axvline(FHZ[k], color="k", lw=.5, alpha=.35, ymax=.08)
    for k in RES["eval", b]: ax[j].axvline(FHZ[k], color="tab:red", lw=.5, alpha=.35, ymin=.92)
    for e in 100 + np.arange(1, 18) * 65 * 900 / 1169: ax[j].axvline(e, color="0.85", lw=.4, zorder=0)
    ax[j].set_ylabel("% of Σ|g·x|"); ax[j].set_title(f"{b}: |grad × input| of soft Dice per bin (ticks: relaxed resonances, black bottom = train, red top = eval; grey = band edges)", fontsize=9); ax[j].legend(loc="upper right")
ax[-1].set_xlabel("Hz"); plt.show()

In [ ]:
# top lines: the 10 largest local maxima of mean |g·x| (≥ 5 Hz apart); distance to nearest relaxed resonance; chance = share of bins within ±3 Hz of any resonance
rows = []
for s in SETS:
    for b in BX:
        g, res = GX[s, b], RES[s, b]; pk = find_peaks(g, distance=7)[0]; top = pk[np.argsort(-g[pk])[:10]]
        dist = np.array([np.abs(FHZ[res] - FHZ[k]).min() for k in top]); near = np.abs(FHZ[:, None] - FHZ[res][None]).min(1) <= 3
        share = g[near].sum() / g.sum()
        rows.append({"set": s, "box": b, "top lines (Hz)": " ".join(f"{FHZ[k]:.0f}" for k in top), "Δ to nearest res (Hz)": " ".join(f"{d:.0f}" for d in dist),
                     "top10 within ±3 Hz": f"{(dist <= 3).sum()}/10", "chance": f"{near.mean():.2f}", "|g·x| mass within ±3 Hz": f"{share:.2f}",
                     "top-10 share of mass": f"{g[top].sum() / g.sum():.3f}"})
TG = pd.DataFrame(rows).set_index(["set", "box"]); pd.set_option("display.max_colwidth", 80); TG

In [ ]:
# |g·x| per band vs the per-band edits (Spearman over 18 bands), and the resonance list itself
for s in SETS:
    print(s, {b: {nm: round(stats.spearmanr(GX[s, b].reshape(18, 65).sum(1), DR[s][b][i])[0], 2) for i, nm in [(0, "pattern"), (2, "shift-2"), (6, "smooth"), (7, "keymask")]} for b in BX})
for b in BX: print(b, "train resonances (Hz):", " ".join(f"{FHZ[k]:.0f}" for k in RES["train", b]))

**FQ3 answer.** _filled in after execution_

## FQ2 · Redundancy vs synergy between bands
- **Pairwise key-masking** of all 153 band pairs (plus the 18 singles, recomputed in the same pass), per sample. Interaction **I(i,j) = Δij − Δi − Δj** on the box-level relative drop (% of clean IoU): **negative = redundant** (dropping both costs less than the sum; one covers for the other), **positive = synergy** (the pair carries information neither carries alone).
- **Shapley values** per band: v(S) = mean IoU of the box's 240 samples with only bands S kept (the rest key-masked); 200 random permutations per box (100 antithetic pairs, one permutation shared by all samples of the box), φᵢ = mean marginal gain when band i is added. v(∅) = no band kept (the freq readout reduces to its bias). By efficiency Σφ = v(all) − v(∅). The forward runs on the kept band tokens only (exactly equal to key-masking, checked in the script).
- Shapley ranking vs single-band masking ranking (Spearman over 18 bands), and the top-K keep curve / **K90** (bands kept for 90% of clean IoU) with the Shapley order, vs masking order from nb136 Q1 (`c|box|band|masking`).

In [ ]:
def inter(s):  # {box: (18,18)} I(i,j) in % of clean IoU, plus singles and pair drops
    f, out = F2[s], {}
    for b in BX:
        m = f["box"] == b; cl = f["clean"][m]; d1 = absd(f["single"][m], cl[:, None]).mean(0) / c0[s][b] * 100
        d2 = absd(np.nan_to_num(f["pair"][m]), cl[:, None, None]).mean(0) / c0[s][b] * 100
        I = d2 - d1[:, None] - d1[None]; np.fill_diagonal(I, np.nan); out[b] = (I, d1, d2)
    return out
IN = {s: inter(s) for s in SETS}
lim = max(np.nanmax(np.abs(IN[s][b][0])) for s in SETS for b in BX)
fig, ax = plt.subplots(2, 3, figsize=(13, 8.4), constrained_layout=True)
for i, s in enumerate(SETS):
    for j, b in enumerate(BX):
        I = IN[s][b][0]; h = ax[i, j].imshow(I, cmap="RdBu_r", vmin=-lim, vmax=lim)
        iu = np.triu_indices(18, 1); v = I[iu]
        ax[i, j].set_title(f"{s} · {b}: {(v < 0).mean() * 100:.0f}% of pairs < 0, mean I {v.mean():+.2f}%", fontsize=9)
        ax[i, j].set_xticks(range(0, 18, 2), [f"{h:.0f}" for h in HZ[::2]], rotation=60, fontsize=7); ax[i, j].set_yticks(range(0, 18, 2), [f"{h:.0f}" for h in HZ[::2]], fontsize=7)
        plt.colorbar(h, ax=ax[i, j], fraction=.046, pad=.02, label="I(i,j), % of clean IoU")
fig.suptitle("FQ2 · pairwise interaction I(i,j) = Δij − Δi − Δj (blue = redundant, red = synergy)"); plt.show()

In [ ]:
rows = []
for s in SETS:
    for b in BX:
        I, d1, d2 = IN[s][b]; v = I[np.triu_indices(18, 1)]; k = np.unravel_index(np.nanargmin(I), I.shape); kk = np.unravel_index(np.nanargmax(I), I.shape)
        nb = np.array([I[i, i + 1] for i in range(17)])
        rows.append({"set": s, "box": b, "Σ singles %": d1.sum().round(1), "mean pair Δij %": d2[np.triu_indices(18, 1)].mean().round(2),
                     "mean I %": v.mean().round(2), "median I %": np.median(v).round(2), "share I<0": (v < 0).mean().round(2), "share |I|>1%": (np.abs(v) > 1).mean().round(2),
                     "most redundant": f"{HZ[k[0]]:.0f}+{HZ[k[1]]:.0f}: {I[k]:+.1f}", "most synergistic": f"{HZ[kk[0]]:.0f}+{HZ[kk[1]]:.0f}: {I[kk]:+.1f}",
                     "adjacent-band mean I %": nb.mean().round(2), "I ~ |i−j| ρ": round(stats.spearmanr(v, np.subtract(*np.triu_indices(18, 1)) * -1)[0], 2)})
pd.DataFrame(rows).set_index(["set", "box"])

In [ ]:
# Shapley vs single-band masking; K90 with each ranking
def k90(c, c_clean, fr=.9): r = np.asarray(c) / c_clean; return int(np.argmax(r >= fr)) + 1 if (r >= fr).any() else 18
def phi_of(f, b, T):  # Shapley from a subset T of the stored permutations: marginal gains from the stored v(S) chains
    P, V, v0 = f[f"perm|{b}"][T], f[f"V|{b}"][T], f[f"v0|{b}"]; g = np.diff(np.c_[np.full(len(V), v0), V], axis=1); out = np.zeros(18)
    for o, gg in zip(P, g): out[o] += gg
    return out / len(T)
fig, ax = plt.subplots(2, 3, figsize=(13, 6.4), constrained_layout=True, sharex=True); rows = []
for i, s in enumerate(SETS):
    f = F2[s]
    for j, b in enumerate(BX):
        phi = 100 * f[f"phi|{b}"] / c0[s][b]; msk = DR[s][b][7]; v0 = f[f"v0|{b}"]
        n = len(f[f"V|{b}"]); half = [phi_of(f, b, np.arange(n // 2)), phi_of(f, b, np.arange(n // 2, n))]; assert np.allclose(phi_of(f, b, np.arange(n)), f[f"phi|{b}"])
        w = .4; ax[i, j].bar(np.arange(18) - w / 2, phi, w, color=BOXC[b], label="Shapley φ"); ax[i, j].bar(np.arange(18) + w / 2, msk, w, color="0.6", label="single-band mask Δ")
        rho = stats.spearmanr(phi, msk)[0]; cs = Q[s][f"c|{b}|band|masking"]; cr = np.mean([Q[s][f"c|{b}|band|random{r}"] for r in range(3)], 0)
        ksh, kms, krd = k90(f[f"c|{b}|shapley"], c0[s][b]), k90(cs, c0[s][b]), k90(cr, c0[s][b])
        ax[i, j].set_title(f"{s} · {b}: ρ(φ, mask) = {rho:.2f}; K90 Shapley {ksh} / mask {kms} / random {krd}", fontsize=9)
        ax[i, j].set_xticks(range(0, 18, 2), [f"{h:.0f}" for h in HZ[::2]], rotation=60, fontsize=8); ax[i, j].set_ylabel("% of clean IoU"); ax[i, j].legend(fontsize=7)
        rows.append({"set": s, "box": b, "v(all)": round(float(c0[s][b]), 3), "v(∅)": round(float(v0), 3), "Σφ %": phi.sum().round(1), "Σ mask Δ %": msk.sum().round(1),
                     "φ top-3 (Hz)": " ".join(f"{HZ[k]:.0f}" for k in np.argsort(-phi)[:3]), "mask top-3 (Hz)": " ".join(f"{HZ[k]:.0f}" for k in np.argsort(-msk)[:3]),
                     "max φ %": phi.max().round(1), "min φ %": phi.min().round(1), "ρ(φ, mask)": round(rho, 2), "ρ(φ 1st, 2nd 100 perms)": round(stats.spearmanr(*half)[0], 2),
                     "K90 Shapley": ksh, "K90 mask": kms, "K90 random": krd,
                     "K50 Shapley": k90(f[f"c|{b}|shapley"], c0[s][b], .5), "K50 mask": k90(cs, c0[s][b], .5)})
fig.suptitle("FQ2 · Shapley value per band vs single-band key-masking drop (% of clean IoU)"); plt.show()
TS = pd.DataFrame(rows).set_index(["set", "box"]); TS

In [ ]:
# top-K keep curves: Shapley order vs masking order vs random (nb136 Q1 curves), % of clean IoU
fig, ax = plt.subplots(1, 6, figsize=(13, 2.8), sharey=True, constrained_layout=True)
for i, (s, b) in enumerate([(s, b) for s in SETS for b in BX]):
    K = np.arange(1, 19); f = F2[s]
    ax[i].plot(K, 100 * f[f"c|{b}|shapley"] / c0[s][b], color=BOXC[b], label="Shapley"); ax[i].plot(Q[s]["KB"], 100 * Q[s][f"c|{b}|band|masking"] / c0[s][b], "--", color="0.3", label="masking")
    ax[i].plot(Q[s]["KB"], 100 * np.mean([Q[s][f"c|{b}|band|random{r}"] for r in range(3)], 0) / c0[s][b], ":", color="0.6", label="random")
    ax[i].axhline(90, color="k", lw=.5); ax[i].set_title(f"{s} · {b}", fontsize=9); ax[i].set_xlabel("K bands kept")
ax[0].set_ylabel("% of clean IoU"); ax[0].legend(fontsize=7); plt.show()

**FQ2 answer.** _filled in after execution_